# 《LangChain PromptTemplate提示词模板实验》

## 一、实验目的
1. 掌握 LangChain 中 PromptTemplate 的基本使用方法  
2. 理解 ChatPromptTemplate 在对话模型中的应用方式  
3. 掌握 Few-shot Prompt（少样本提示词）的构建方法  
4. 理解 MessagesPlaceholder 在多轮对话中的作用  
5. 熟悉 format / invoke / partial 等提示词执行方式  


## 二、实验环境
- 系统：Windows 10  
- Python版本：3.10  
- 虚拟环境：Miniconda  
- 开发工具：VS Code / Jupyter Notebook  
- 依赖库：langchain、langchain-core、langchain-community、faiss-cpu、dashscope、python-dotenv  
- 模型：通义千问（Qwen）、OpenAI兼容模型  




## 三、实验原理

### 3.1 PromptTemplate介绍

在与大语言模型交互时，通常不会直接将用户的原始输入直接传递给大模型，而是会先进行一系列包装、组织和格式化操作。这样做的目的是：更清晰地表达用户意图，更好地利用模型能力。

这套结构化的提示词构建方式，就是 LangChain 中的 提示词模板（PromptTemplate）。对于 LLM 应用来说，好的提示词就是成功的一半。更多提示词技巧可参考文档：https://www.cuiliangblog.cn/detail/section/228046450。

LangChain 提示词官方文档参考：https://reference.langchain.com/python/langchain_core/prompts/

### 3.2 提示词模板分类

LangChain 提供了多种不同的提示词模板，下面介绍几种常用的提示词模板：

- PromptTemplate：文本生成模型提示词模板，用字符串拼接变量生成提示词
- ChatPromptTemplate：聊天模型提示词模板，适用于如 gpt-3.5-turbo、gpt-4 等聊天模型
- HumanMessagePromptTemplate：人类消息提示词模板
- SystemMessagePromptTemplate：系统消息提示词模板
- FewShotPromptTemplate：少样本学习提示词模板， 构建一个 Prompt，其中包含多个 示例，可以 自动将这些示例格式化并插入到主 Prompt 中

### 3.3 文本提示词模板

PromptTemplate 针对文本生成模型的提示词模板，也是LangChain提供的最基础的模板，通过格式化字符串生成提示词，在执行invoke时将变量格式化到提示词模板中

主要参数：

- template：定义提示词模板的字符串，其中包含文本和变量占位符（如{name}） ；
- input_variables： 列表，指定了模板中使用的变量名称，在调用模板时被替换；
- partial_variables：字典，用于定义模板中一些固定的变量名。这些值不需要再每次调用时被替换。

函数介绍：
- format()：给input_variables变量赋值，并返回提示词。利用format() 进行格式化时就一定要赋值，否则会报错。当在template中未设置input_variables，则会自动忽略。

### 3.4 对话提示词模板

ChatPromptTemplate 是专为聊天模型（如 gpt-3.5-turbo、gpt-4 等）设计的提示词模板，它支持构造多轮对话的消息结构，每条消息可指定角色（如系统、用户、AI）。

特点：

- 支持 System / Human / AI 等不同角色的消息模板
- 对话历史维护
- 参数类型：列表参数格式是tuple类型（ role :str content :str 组合最常用）
- 元组的格式为：(role: str | type, content: str | list[dict] | list[object])

其中 role 是：字符串（如 “system” 、“human” 、“ai” ）

### 3.5 少样本提示词模板

FewShotPromptTemplate 用于：

- 构建一个 Prompt，其中包含多个 示例（examples）；
- 自动将这些示例格式化并插入到主 Prompt 中；
- 实现 Few-Shot Prompting 方式，以增强大模型在特定任务（如分类、问答、翻译等）上的表现。

它通常由以下几部分构成：

1. examples：少量的人工示例（dict 列表）；
2. example_prompt：如何格式化每个示例（使用 PromptTemplate）；
3. prefix：示例之前的文字说明（可选）；
4. suffix：用户真正的问题模板；
5. input_variables：最终 suffix 中需要传入的变量。

## 四、实验内容

### 4.1 PromptTemplate基础使用

PromptTemplate针对文本生成模型的提示词模板，是LangChain提供的最基础的模板，通过格式化字符串生成提示词，在执行invoke时将变量格式化到提示词模板中。

In [1]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    template="你是一个专业的{role}工程师，请回答：{question}",
    input_variables=["role", "question"]
)

prompt = template.format(role="python开发", question="冒泡排序怎么写？")
print(prompt)

你是一个专业的python开发工程师，请回答：冒泡排序怎么写？


### 4.2 部分提示词模板（Partial）

部分提示词，允许预先固定部分变量，而保留其他变量在后续动态填充。例如：先预设系统参数，然后等用户输入后再补齐提示词模板。

In [2]:
from datetime import datetime
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "现在时间是：{time}，问题：{question}",
    partial_variables={"time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
)

prompt = template.format(question="今天是几号？")
print(prompt)

现在时间是：2026-05-25 17:27:08，问题：今天是几号？


### 4.3 组合提示词模板

通过将多个子提示按一定逻辑顺序或层级组合起来，形成一个复杂任务的整体Prompt。例如实现多消息对话、多阶段任务、多输入源组合等场景。

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_a = PromptTemplate.from_template("请介绍{topic}\n")
prompt_b = PromptTemplate.from_template("限制在{length}字以内")

prompt = (prompt_a + prompt_b).format(topic="LangChain", length=20)
print(prompt)

### 4.4 ChatPromptTemplate对话模板

ChatPromptTemplate是专为聊天模型（如gpt-3.5-turbo、gpt-4等）设计的提示词模板，支持构造多轮对话的消息结构，每条消息可指定角色（如系统、用户、AI）。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是{role}"),
    ("human", "{question}")
])

result = chat_prompt.invoke({
    "role": "Python工程师",
    "question": "冒泡排序怎么写"
})

print(result.to_string())

### 4.5 format_messages与format_prompt

format_messages将模板变量替换后直接生成消息列表（List[BaseMessage]），包含SystemMessage、HumanMessage、AIMessage。format_prompt生成PromptValue对象，可通过.to_string()转成文本，或.to_messages()转成消息列表。

In [ ]:
messages = chat_prompt.format_messages(
    role="Python工程师",
    question="你好"
)
print(messages)

prompt_value = chat_prompt.format_prompt(
    role="Python工程师",
    question="你好"
)

print(prompt_value.to_messages())

### 4.6 Few-shot PromptTemplate

FewShotPromptTemplate用于构建包含多个示例的Prompt，自动将示例格式化并插入到主Prompt中，以增强大模型在特定任务（如分类、问答、翻译等）上的表现。通常由examples、example_prompt、prefix、suffix、input_variables几部分构成。

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

examples = [
    {"input": "北京下雨吗", "output": "北京"},
    {"input": "上海热吗", "output": "上海"},
]

example_prompt = PromptTemplate(
    template="输入:{input}\n输出:{output}",
    input_variables=["input", "output"]
)

few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="输入:{input}\n输出:",
    input_variables=["input"]
)

print(few_shot.format(input="天津今天刮风吗"))

### 4.7 FewShotChatMessagePromptTemplate

FewShotChatMessagePromptTemplate专门为聊天对话场景设计的少样本提示模板，继承自FewShotPromptTemplate，但针对聊天消息格式进行了优化。自动将示例格式化为聊天消息（HumanMessage/AIMessage），输出结构化聊天消息并保留对话轮次结构。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

examples = [
    {"input": "1x2", "output": "2"},
    {"input": "2x2", "output": "4"},
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}是多少"),
    ("ai", "{output}")
])

few_shot = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一名数学专家")
]) + few_shot + ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

print(final_prompt.format(question="3x2"))

### 4.8 MessagesPlaceholder多轮对话记忆

在ChatPromptTemplate中添加MessagesPlaceholder占位符，可以在调用invoke时动态插入消息。适用于不确定消息何时生成、也不确定要插入几条消息的场景，比如在提示词中添加聊天历史记忆。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("memory"),
    ("system", "你是Python专家"),
    ("human", "{question}")
])

result = prompt.invoke({
    "memory": [
        HumanMessage("我叫张三"),
        AIMessage("你好张三")
    ],
    "question": "我叫什么名字？"
})

print(result.to_string())

## 五、实验总结

1. 实际创建了 PromptTemplate 和 ChatPromptTemplate 两种模板，理解了文本模板与对话模板的使用场景区别
2. 通过 format、invoke、partial 三种方法的对比实验，掌握了它们各自的返回类型和适用场景
3. 组合多个子模板实现复杂提示词拼接，体验了模板复用的便利性
4. 构建 FewShotPromptTemplate 和 FewShotChatMessagePromptTemplate，理解了少样本提示在提升模型输出质量上的作用
5. 使用 MessagesPlaceholder 实现多轮对话记忆注入，掌握了聊天历史上下文的管理方式